# 赋值更新与写时复制

学习目标：创建和更新表格列，按标签合并修订值，并区分别名、派生对象和写时复制的修改范围。

前置知识：Series 与 DataFrame、标签对齐、loc、Python 赋值与对象引用。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5；按 pandas 3 的 Copy-on-Write 行为编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

所有表格均为自制输入，后续单元沿用首次导入的 pd；只有连续操作同一张表的小节复用前面的变量。

## 1 创建列与条件赋值

用三个商品的单价和数量计算小计，再把数量为 0 的商品标为暂缺。示例金额使用整数“分”，没有货币换算或舍入。

df['新列名'] = 值 会在表中创建列，列名已经存在时则重新赋值。对原表的部分位置更新，用一次 loc 同时给出行条件和列名。

In [1]:
import pandas as pd

items = pd.DataFrame({"unit_price": [120, 250, 80], "quantity": [2, 0, 3]},
                     index=["A", "B", "C"])
items["subtotal"] = items["unit_price"] * items["quantity"]
items["status"] = "有货"
items.loc[items["quantity"] == 0, "status"] = "暂缺"
print(items)  # 小计依次为 240、0、240 分；只有 B 为暂缺。
print(items.shape, items.index.tolist())  # (3, 4)，标签仍为 A、B、C。

   unit_price  quantity  subtotal status
A         120         2       240     有货
B         250         0         0     暂缺
C          80         3       240     有货
(3, 4) ['A', 'B', 'C']


## 2 按标签赋值

将 Series 赋给整列时，pandas 按行标签匹配，不按 Series 中的出现顺序填入。列表则按当前行顺序提供数值，长度要与行数匹配。

下面两份数据数值顺序相同，但 Series 的标签是 C、A、B；核对两列在 A 行的区别。

In [2]:
table = pd.DataFrame({"quantity": [2, 4, 6]}, index=["A", "B", "C"])
correction = pd.Series([30, 10, 20], index=["C", "A", "B"])
table["by_label"] = correction
table["by_position"] = [30, 10, 20]
print(table)  # by_label 为 10、20、30；by_position 为 30、10、20。
print(table.dtypes)  # 三列均为整数，标签对齐没有引入缺失。

   quantity  by_label  by_position
A         2        10           30
B         4        20           10
C         6        30           20
quantity       int64
by_label       int64
by_position    int64
dtype: object


## 3 用 assign 返回新表

assign 返回含有原列和新增列的新 DataFrame；同名列在结果中被覆盖。它适合保留原表、继续处理结果的场合，直接列赋值则修改调用的表。

先用已经得到的 Series 计算新列，避免在一个表达式中隐藏多步处理。

In [3]:
items = pd.DataFrame({"price": [100, 200], "quantity": [2, 3]}, index=["A", "B"])
result = items.assign(subtotal=items["price"] * items["quantity"])
print(result)  # subtotal 为 200、600 分。
print(items.columns.tolist(), result.columns.tolist())
# 原表只有 price、quantity；结果多出 subtotal。
print(result is items)  # False：结果是另一个 DataFrame 对象。

   price  quantity  subtotal
A    100         2       200
B    200         3       600
['price', 'quantity'] ['price', 'quantity', 'subtotal']
False


assign 也接受函数。函数接收正在生成的表，返回待加入的列；参数按给定顺序计算，后面的函数可以引用刚创建的列。函数不得修改接收的表。

下面 lambda table 表示接收 table 并计算一个表达式的匿名函数；total 在 subtotal 之后计算，固定运费为每行 50 分。

In [4]:
items = pd.DataFrame({"price": [100, 200], "quantity": [2, 3]}, index=["A", "B"])
result = items.assign(
    subtotal=lambda table: table["price"] * table["quantity"],
    total=lambda table: table["subtotal"] + 50,
)
print(result)  # subtotal 为 200、600；total 为 250、650。
print(items.shape, result.shape)  # (2, 2) 与 (2, 4)。

   price  quantity  subtotal  total
A    100         2       200    250
B    200         3       600    650
(2, 2) (2, 4)


## 4 插入与移除列

insert 在指定列位置插入新列，直接修改原表。loc 参数是从 0 开始的插入位置，不是列标签；默认不允许重复列名。若传入 Series，仍然按行标签对齐。

需要控制新列在表中出现的位置时用 insert；只想加在末尾时，普通列赋值更直接。

In [5]:
table = pd.DataFrame({"value": [10, 20]}, index=["A", "B"])
table.insert(0, "source", ["室内", "室外"])
print(table)  # source 位于 value 前面。
print(table.columns.tolist())  # ['source', 'value']
try:
    table.insert(0, "value", [30, 40])
except ValueError:
    print("已有同名列，默认拒绝插入")
else:
    raise AssertionError("重复列名应被拒绝")

  source  value
A     室内     10
B     室外     20
['source', 'value']
已有同名列，默认拒绝插入


drop(columns=...) 默认返回移除指定列后的新表；drop(index=...) 按行标签移除行。pop 则从原表移除一个列，并把该列作为 Series 返回。

下面的列名唯一。若只要删列后的新表，用 drop；若还要取走这一列的内容，用 pop。对缺失标签，二者默认都会报 KeyError。

In [6]:
table = pd.DataFrame({"value": [10, 20], "note": ["复核", "正常"]}, index=["A", "B"])
without_note = table.drop(columns=["note"])
print(without_note.shape, table.shape)  # (2, 1)、(2, 2)：原表尚未移除列。
removed = table.pop("note")
print(removed)  # Series，保留 A、B 行标签。
print(table.columns.tolist())  # 原表现在只有 value。
print(without_note.drop(index="A"))  # 按标签移除 A，只留下 B。

(2, 1) (2, 2)
A    复核
B    正常
Name: note, dtype: str
['value']
   value
B     20


## 5 用 update 合并修订值

update 根据行、列标签，把修订表中的非缺失值写回调用表。它直接修改原表，返回 None；只保留原表的行列范围，不因修订表多出标签就扩展表格。

下面修订表的行顺序为 C、A、D，原表为 A、B、C。C 的读数改成 99，A 的修订值缺失所以保留原值，D 行和 extra 列不在原表范围内。

In [7]:
original = pd.DataFrame({"reading": [10.0, 20.0, 30.0]}, index=["A", "B", "C"])
patch = pd.DataFrame({"reading": [99.0, float("nan"), 777.0], "extra": [1, 2, 3]},
                     index=["C", "A", "D"])
returned = original.update(patch)
print(original)  # A=10、B=20、C=99；没有 D 行或 extra 列。
print(original.index.tolist(), original.columns.tolist(), original.shape)
print(returned)  # None：不要写 original = original.update(patch)。
print(original.dtypes)  # reading 仍为 float64。

   reading
A     10.0
B     20.0
C     99.0
['A', 'B', 'C'] ['reading'] (3, 1)
None
reading    float64
dtype: object


默认 overwrite=True 会覆盖双方都有数值的位置。overwrite=False 只给原表的缺失位置补值；修订表自己的缺失值仍不参与更新。

以下为选学参数。两份结果都从同一原始输入创建，便于比较覆盖规则。

In [8]:
original = pd.DataFrame({"reading": [10.0, float("nan"), 30.0]}, index=["A", "B", "C"])
patch = pd.DataFrame({"reading": [90.0, 20.0]}, index=["A", "B"])
overwritten = original.copy()
filled = original.copy()
overwritten.update(patch)
filled.update(patch, overwrite=False)
print(overwritten["reading"].tolist())  # [90.0, 20.0, 30.0]
print(filled["reading"].tolist())  # [10.0, 20.0, 30.0]：只填 B。
print(original["reading"].isna().tolist())  # [False, True, False]：原输入未改。

[90.0, 20.0, 30.0]
[10.0, 20.0, 30.0]
[False, True, False]


errors='raise' 在双方同位置都有非缺失数据时抛出 ValueError，即使两个值相同也算重叠。需要保护原始输入时，在副本上尝试更新，检查成功后再使用结果；不能把 update 当作数据库事务。

修订表的行索引不能重复。对重复修订记录，应先决定如何选取，不能交给 update 猜测优先级。

In [9]:
original = pd.DataFrame({"reading": [10.0, 20.0]}, index=["A", "B"])
patch = pd.DataFrame({"reading": [10.0]}, index=["A"])
candidate = original.copy()
try:
    candidate.update(patch, errors="raise")
except ValueError:
    print("双方都有值，触发重叠检查")  # 相同的 10.0 也属于非缺失重叠。
else:
    raise AssertionError("重叠数据应被报告")
print(original)  # 原始输入仍是 A=10、B=20。

双方都有值，触发重叠检查
   reading
A     10.0
B     20.0


## 6 别名与派生对象

other = table 只为同一个 Python 对象增加名字，不产生另一张表。用任一个名字修改对象，另一个名字都会看到变化。

pandas 3 的写时复制（Copy-on-Write，CoW）是默认且唯一模式，但不会改变这种 Python 别名关系。

In [10]:
table = pd.DataFrame({"value": [10, 20]}, index=["A", "B"])
alias = table
alias.loc["A", "value"] = 99
print(alias is table)  # True
print(table)  # A 的 value 已变为 99。

True
   value
A     99
B     20


选择列或切片得到的是派生对象。在 pandas 3 中，通过 pandas 赋值修改派生对象，不会回写原表；修改原表也不会连带修改已有派生对象。

底层数据可能暂时共享，需要写入时再复制，这是 CoW 的实现方式。判断教学示例的修改范围，先看操作针对哪个 pandas 对象，不必猜测内部数据块。

In [11]:
table = pd.DataFrame({"value": [10, 20]}, index=["A", "B"])
column = table["value"]
column.loc["A"] = 99
table.loc["B", "value"] = 88
print(column.tolist())  # [99, 20]：修改原表 B 不会改变此前选出的列。
print(table["value"].tolist())  # [10, 88]：修改列对象 A 没有回写原表。

[99, 20]
[10, 88]


copy(deep=True) 是默认的显式复制；copy(deep=False) 创建另一个对象，暂时共享底层数据，并在 pandas 写入时隔离变化。pandas 3 中，不能把 deep=False 理解为“后续修改会同步到原表”。

In [12]:
table = pd.DataFrame({"value": [10, 20]}, index=["A", "B"])
shallow = table.copy(deep=False)
deep = table.copy()
shallow.loc["A", "value"] = 90
deep.loc["B", "value"] = 80
print(table["value"].tolist())  # [10, 20]
print(shallow["value"].tolist(), deep["value"].tolist())  # [90, 20]、[10, 80]
print(shallow is table, deep is table)  # False False

[10, 20]
[90, 20] [10, 80]
False False


deep=True 不会递归复制 object 列里的 Python 可变对象。下面对列表调用 append，并没有进行 pandas 的单元格赋值，所以列表仍可能被双方引用。

入门表格优先保存标量字段；必须存储嵌套可变对象时，另行决定这些对象的复制规则。

In [13]:
table = pd.DataFrame({"values": [[1, 2], [3, 4]]})
copied = table.copy(deep=True)
copied.at[0, "values"].append(9)
print(table.at[0, "values"], copied.at[0, "values"])  # 都是 [1, 2, 9]。
print(table.at[0, "values"] is copied.at[0, "values"])  # True：内部列表是同一个对象。
copied.at[0, "values"] = [7]
print(table.at[0, "values"], copied.at[0, "values"])  # 换槽位后分别为 [1, 2, 9]、[7]。

[1, 2, 9] [1, 2, 9]
True
[1, 2, 9] [7]


## 7 链式赋值与列上的 inplace

table['value'][条件] = ... 经过两次选择，把值写入中间对象，不能更新原表。pandas 3 会给出 ChainedAssignmentError 警告；名称中虽有 Error，它在这里是警告类别。

下面用局部 warnings 上下文收集并检查这个预期警告，没有关闭全局警告。随后用一次 loc 把同一任务写到原表。

In [14]:
import warnings

table = pd.DataFrame({"value": [10, 20, 30]}, index=["A", "B", "C"])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", pd.errors.ChainedAssignmentError)
    table["value"][table["value"] > 15] = 0
print([warning.category.__name__ for warning in caught])
assert any(issubclass(warning.category, pd.errors.ChainedAssignmentError) for warning in caught)
print(table["value"].tolist())  # [10, 20, 30]：链式赋值没有改原表。
table.loc[table["value"] > 15, "value"] = 0
print(table["value"].tolist())  # [10, 0, 0]：直接修改原表。

['ChainedAssignmentError']
[10, 20, 30]
[10, 0, 0]


对临时选出的列调用 inplace=True 方法，也不能据此修改原表。inplace 指向实际接收方法的那个对象，不会穿过派生关系回写父表。

需要替换一列中的值时，可把方法返回值显式赋回该列。下面用 replace 演示，它把指定旧值替换为新值。

In [15]:
table = pd.DataFrame({"value": [10, 20]}, index=["A", "B"])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", pd.errors.ChainedAssignmentError)
    table["value"].replace(10, 99, inplace=True)
print([warning.category.__name__ for warning in caught])
assert any(issubclass(warning.category, pd.errors.ChainedAssignmentError) for warning in caught)
print(table["value"].tolist())  # [10, 20]：原表不变。
table["value"] = table["value"].replace(10, 99)
print(table["value"].tolist())  # [99, 20]：显式赋回才修改原表。

['ChainedAssignmentError']


[10, 20]
[99, 20]


## 8 NumPy 转换与只读共享

to_numpy 不一定复制数据。对于由单个 NumPy 数据块组成的同类型数值表，结果可能共享存储，此时 pandas 把返回数组设为只读，以保护 CoW 约定。不同类型的表可能需要转换和复制，不能把这个例子推广成“所有结果都共享”。

若需要在 NumPy 中独立修改，使用 to_numpy(copy=True)。不要通过强行打开写入标志绕过原表的保护。

In [16]:
import numpy as np

table = pd.DataFrame({"left": [1, 2], "right": [3, 4]})
borrowed = table.to_numpy()
editable = table.to_numpy(copy=True)
print(borrowed.shape, borrowed.dtype, borrowed.flags.writeable)  # (2, 2)、int64、False。
print(editable.flags.writeable, np.shares_memory(borrowed, editable))  # True False
try:
    borrowed[0, 0] = 99
except ValueError:
    print("共享结果只读，赋值被拒绝")
else:
    raise AssertionError("本例共享数组应为只读")
editable[0, 0] = 99
print(editable[0, 0], table.loc[0, "left"])  # 99 与 1，副本修改不影响表格。

(2, 2) int64 False
True False
共享结果只读，赋值被拒绝
99 1


## 本章小结

（1）列赋值和 loc 可以直接更新指定表格；Series 值按标签对齐，列表按当前顺序提供值。

（2）assign 和默认 drop 返回新表，insert、pop、update 修改调用对象；选择前先确定是否需要保留原表。

（3）update 只更新原表范围内的匹配位置，默认使用修订表的非缺失值；覆盖和冲突规则可以显式设置。

（4）别名仍指向同一个对象。pandas 3 的派生对象在 pandas 赋值时隔离修改，但 object 内部可变对象需要另行考虑。

（5）链式赋值和临时列上的 inplace 不能更新原表。共享的 NumPy 结果可能只读，需要独立修改时显式复制。

## 练习

（1）给商品表增加小计列，金额单位为分。将数量小于 1 的商品状态设为“待补货”，其余为“可用”。要求直接修改原表，不能链式赋值。

In [17]:
items = pd.DataFrame({"price": [150, 300, 90], "quantity": [2, 0, 4]},
                     index=["X", "Y", "Z"])
# 在此创建 subtotal 和 status，用 loc 条件赋值。
# 检查：小计为 300、0、360 分；仅 Y 待补货，原行标签和顺序不变。

（2）先预测三个对象最后的内容，再运行核对。解释 alias 与 selected 的不同之处；如果要求 alias 也能独立修改，该改哪一句？

In [18]:
table = pd.DataFrame({"value": [1, 2]}, index=["A", "B"])
alias = table
selected = table["value"]
alias.loc["A", "value"] = 8
selected.loc["B"] = 9
print(table["value"].tolist())
print(alias["value"].tolist())
print(selected.tolist())
# 在此写出修改对象的判断，并按独立修改要求重做。

[8, 2]
[8, 2]
[1, 9]


（3）现在修订表只能补齐原表的缺失值，不能覆盖已有读数，也不能新增设备。选择 update 的参数并解释原因；核对乱序标签和新增 D 行的处理。

In [19]:
original = pd.DataFrame({"reading": [10.0, float("nan"), 30.0]}, index=["A", "B", "C"])
patch = pd.DataFrame({"reading": [200.0, 99.0, 400.0]}, index=["B", "A", "D"])
# 在此在 original 的副本上完成更新，打印索引、shape、dtype 与结果。
# 检查：原输入保留；结果 A=10、B=200、C=30，不能出现 D。

（4）对数值表转换后的 NumPy 数组加 100，要求原表保持不变。选择转换方式并检查；说明为什么 copy=False 不是零复制的保证。

In [20]:
table = pd.DataFrame({"first": [1, 2], "second": [3, 4]})
# 在此取得可独立修改的数组，对全部元素加 100。
# 检查：打印数组和原表；原表的标签、dtype 和数值不变。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | pandas 3.0.6：[Indexing](https://pandas.pydata.org/docs/user_guide/indexing.html) 的 Setting with enlargement、Boolean indexing 及标签赋值；[assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) 的返回对象、callable 与 Notes；[insert](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.insert.html) 的位置、重名和 Series 对齐；[drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)、[pop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pop.html) 的返回与修改行为；[update](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.update.html) 的 join、overwrite、errors、重复索引 Notes；[Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html) 的 Migrating、Description、Chained Assignment、Read-only NumPy arrays；[copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) 的 deep=False 及非递归复制 Notes；[to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html) 的 copy 和类型转换条件；[ChainedAssignmentError](https://pandas.pydata.org/docs/reference/api/pandas.errors.ChainedAssignmentError.html)（页面标识 3.0.5）的警告类别与 CoW 说明，行为另按 3.0.6 CoW 指南核查。 |
| Python 官方文档 | Python 3.12：[warnings](https://docs.python.org/3.12/library/warnings.html) 的 Testing Warnings、catch_warnings 和 simplefilter：局部收集并检查预期警告。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[indexing](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/indexing.rst)、[copy_on_write](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/copy_on_write.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |